**О проекте:**  
Данный проект выполнен в рамках практического обучения. Базовый пайплайн обработки данных (EDA, первичная очистка) разрабатывался под менторством преподавателя (я реализовывал код step-by-step в ходе урока).

**Моя самостоятельная часть работы (финальное задание) включала:**
* Обработку признака living_region
* Самостоятельную генерацию новых признаков
* Обучение модели машинного обучения на тренировочных данных.
* Подбор гиперпараметров
* Получите предсказания модели на тестовых данных test_data


##File download

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
pip install category_encoders

In [ ]:
! pip install wldhx.yadisk-direct
! curl -L $(yadisk-direct https://disk.yandex.com/d/sknuSa3xoNBsDw) -o bank-issues-data.zip
! unzip -qq bank-issues-data.zip

In [ ]:
! unzip -qq bank-issues-data.zip

##Data processing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
train_data = pd.read_csv('./bank-issues-data/train.csv')
test_data = pd.read_csv('./bank-issues-data/test.csv')

In [ ]:
train_data

Поля данных:
- **client_id** — Уникальный идентификатор клиента
- **gender** — Пол
- **age** — Возраст (в годах)
- **marital_status** — Семейное положение.
    Возможные значения:
    - UNM : Холост/не замужем
    - DIV : Резведен (а)
    - MAR : Женат/замужем
    - WID : Вдовец, вдова
    - CIV : Гражданский брак
- **job_position** — Работа.
    Возможные значения:
    - SPC : Неруководящий сотрудник - специалист
    - DIR : Руководитель организации
    - HSK : Домохозяйка
    - WOI : Работает на ИП
    - WRK : Неруководящий сотрудник - рабочий
    - ATP : Неруководящий сотрудник - обслуживающий персонал
    - WRP : Работающий пенсионер
    - UMN : Руководитель подразделения
    - NOR : Не работает
    - NS : Пенсионер
    - BIS : Собственный бизнес
    - INP : Индивидуальный предприниматель
- **credit_sum** — Сумма кредита
- **credit_month** — Срок кредитования в месяцах
- **tariff_id** — Номер предлагаемого тарифа
- **education** — Тип образования.
    Возможные знаяения:
    - SCH : Начальное, среднее
    - PGR : Второе высшее
    - GRD : Высшее
    - UGR : Неполное высшее
    - ACD : Ученая степень
- **living_region** — Регион проживания
- **monthly_income** — Зарплата в месяц
- **credit_count** — Количество кредитов у клиента
- **overdue_credit_count** — Количество просроченных кредитов клиента
- **open_account_flag** — Целевая переменная -- выберет клиент наш банк или нет

In [ ]:
y_train = train_data['open_account_flg']
X_train = train_data.drop(columns='open_account_flg')

train_data.info()

In [ ]:
columns_to_drop = ['client_id']

train_data.drop(columns=columns_to_drop, inplace=True, errors='ignore')
test_data.drop(columns=columns_to_drop, inplace=True, errors='ignore')

display(train_data.head())

##Заполняю нулевые данные

In [ ]:
np.unique(train_data['credit_count'], return_counts=True)

In [ ]:
train_data['job_position'].dtype

In [ ]:
#выбираем все столбцы с типом данных int или float, чтобы использовать ее в KNN inputer
num_col = [column for column in train_data.columns if train_data[column].dtype != 'O']
num_col

In [ ]:
from sklearn.impute import KNNImputer

knn_inputer = KNNImputer(n_neighbors=2)
knn_inputer.fit(train_data[num_col])

transformed_overdue_credit_count = knn_inputer.transform(train_data[num_col])[:, 8]
train_data['overdue_credit_count'] = transformed_overdue_credit_count

In [ ]:
train_data['overdue_credit_count'].isna().any()

##Обработка категориальных признаков

In [ ]:
categorical_columns = [column for column in train_data.columns if train_data[column].dtype == 'O']
categorical_columns += ['tariff_id'] #tariff_id также является категориальной переменной не смотря на числовой тип данных
categorical_columns = [x for x in categorical_columns if x != 'living_region']
np.array(categorical_columns)

###One hot encoder

In [ ]:
# использую именно этот метод, так как он не создаёт мнимых иерархий внутренних признакво,
# что делает его лучше чем label encoding.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(drop='if_binary', sparse_output=False)
ohe.fit(train_data[categorical_columns])

new_categorical_col = ohe.transform(train_data[categorical_columns])
new_categorical_col

In [ ]:
ohe.get_feature_names_out()

In [ ]:
new_train_columns = pd.DataFrame(new_categorical_col, columns=ohe.get_feature_names_out())
train_data = train_data.drop(columns=categorical_columns)
train_data = pd.concat([train_data, new_train_columns], axis=1)
train_data.head()

In [ ]:
# дополнительно обрабатываем living_region:
# для этого: города разделим по названию (убрав вариации г; г.; город; и тд.)
# те города которые встречаются меньше 10 раз мы удаляем и не считываем (тем самым убираем выборсы)
# если в тест данных встречаются регионы которых нет в трейн данных мы их удаляем

import re

In [ ]:
def clean_region(text):
    if pd.isna(text):
        return 'unknown'
    text = text.lower().strip()
    text = re.sub(r'\b(г|гор|город|п|пос|село|деревня|д)\.?\s+', '', text)
    return text

train_data['living_region'] = train_data['living_region'].apply(clean_region)
test_data['living_region'] = test_data['living_region'].apply(clean_region)

region_counts = train_data['living_region'].value_counts()
valid_regions = region_counts[region_counts >= 10].index

train_data = train_data[train_data['living_region'].isin(valid_regions)].reset_index(drop=True)

test_data = test_data[test_data['living_region'].isin(valid_regions)].reset_index(drop=True)

print(f"Уникальных регионов после обработки: {len(train_data['living_region'].unique())}")
display(train_data['living_region'].value_counts().head())

In [ ]:
import category_encoders as ce

target_encoder = ce.TargetEncoder(cols=['living_region'], smoothing=10)

# Исправлено название колонки с 'open_account_flag' на 'open_account_flg'
train_data['living_region_encoded'] = target_encoder.fit_transform(
    train_data['living_region'],
    train_data['open_account_flg']
)

test_data['living_region_encoded'] = target_encoder.transform(test_data['living_region'])

train_data = train_data.drop('living_region', axis=1)
test_data = test_data.drop('living_region', axis=1)

train_data['living_region_encoded']

##Обучение модели на тренировочных данных

In [ ]:
print("Есть ли таргет в train_data?", 'open_account_flg' in train_data.columns)
print("Есть ли таргет в test_data?", 'open_account_flg' in test_data.columns)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# подготовка train
X = pd.get_dummies(train_data.drop(['open_account_flg', 'overdue_credit_count'], axis=1))
y = train_data['open_account_flg']

# обучение модели
rf_model = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X, y)
print("Модель успешно обучена!")

# подготовка test
if 'id' in test_data.columns:
    ids = test_data['id']

X_test_final = pd.get_dummies(
    test_data.drop(columns=['open_account_flg', 'overdue_credit_count'], errors='ignore')
)
# выравниваем признаки
X_test_final = X_test_final.reindex(columns=X.columns, fill_value=0)


In [ ]:
y_pred_proba = rf_model.predict_proba(X_test_final)[:, 1]

In [ ]:
from sklearn.metrics import roc_auc_score

y_pred_train = rf_model.predict_proba(X)[:,1]

roc_auc = roc_auc_score(y, y_pred_train)
print("ROC-AUC:", roc_auc)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = pd.get_dummies(train_data.drop('open_account_flg', axis=1))
y = train_data['open_account_flg']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

rf_model.fit(X_train, y_train)

y_pred_val = rf_model.predict_proba(X_val)[:,1]

roc_auc = roc_auc_score(y_val, y_pred_val)
print("Validation ROC-AUC:", roc_auc)


In [ ]:
corr = train_data.corr(numeric_only=True)['open_account_flg'].sort_values(ascending=False)
print(corr.head(15))


### Исправление утечки данных и получение реалистичных метрик
Удалим признак `overdue_credit_count`, так как он вызывает аномально высокий результат.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X_leakproof = X.drop(columns=['overdue_credit_count'], errors='ignore')
y_leakproof = y

X_train_lp, X_val_lp, y_train_lp, y_val_lp = train_test_split(
    X_leakproof, y_leakproof,
    test_size=0.2,
    random_state=42,
    stratify=y_leakproof)

rf_final = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1)

rf_final.fit(X_train_lp, y_train_lp)

y_pred_lp = rf_final.predict_proba(X_val_lp)[:, 1]
real_roc_auc = roc_auc_score(y_val_lp, y_pred_lp)

print(f'Реалистичный Validation ROC-AUC: {real_roc_auc:.4f}')

В ходе самостоятельной части проекта была решена задача прогнозирования отклика клиента на кредитное предложение банка (Propensity to Buy).

**1. Выбор и оценка модели:**
Наилучший результат показала модель **RandomForestClassifier**.
* **ROC-AUC (валидация):** 0.7470 — после устранения утечки данных модель показывает стабильное и реалистичное качество классификации.
* **Precision (Точность):** Не рассчитывалась в текущем коде (требует подбора порога).
* **Recall (Полнота):** Не рассчитывалась в текущем коде (требует подбора порога).

**2. Feature Importance:**

После исключения признака с утечкой (`overdue_credit_count`), основными предикторами стали:
1. **tariff_id_1.32** (Специфический тарифный план)
2. **living_region_encoded** (Регион проживания с учетом Target Encoding)
3. **education_SCH** (Тип образования)

**Резюме для бизнеса:**
Внедрение данной модели позволит отделу маркетинга/продаж отказаться от "веерного" обзвона всей базы. Использование предсказаний модели (таргетирование на топ-N% клиентов с наивысшей вероятностью отклика) позволит существенно снизить затраты на коммуникацию и повысить общую конверсию кампании (Lead Conversion Rate). Текущая метрика ROC-AUC 0.7470 является надежной основой для пилотного запуска.